In [ ]:
import pandas as pd
df = pd.read_csv('output/recordings_subset_with_supergenres.csv')
df['youtube_id'] = ''
df[['ISRC', 'youtube_id']].to_csv('output/isrcs.csv', index=False)

In [ ]:
import csv
import json
import random
import urllib.parse
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from youtube_search import YoutubeSearch

SOURCE_CSV_PATH = "output/isrcs.csv"
PROXY_FILE = "proxies.txt"

# batch tuning
BATCH_SIZE = 300
MAX_WORKERS = 6
SAVE_EVERY = 20
TIMEOUT = 8
RETRIES = 0
SEARCH_MAX_RESULTS = 10

# this matching rule ensures that only videos that are also part of Youtube Music's results are accepted
# without it, videos that are not related to music may often be included as top results for some ISRCs
REQUIRED_URL_SUFFIX_TOKEN = "start_radio=1"

def load_proxies(proxy_file):
    """Load proxies in ip:port:user:pass format for requests."""
    proxies = []
    with open(proxy_file, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue

            parts = line.split(":")
            if len(parts) == 4:
                host, port, username, password = parts
                username = urllib.parse.quote(username, safe="")
                password = urllib.parse.quote(password, safe="")
                proxy_url = f"http://{username}:{password}@{host}:{port}"
            elif line.startswith(("http://", "https://", "socks5://", "socks5h://")):
                proxy_url = line
            else:
                print(f"Skipping invalid proxy line: {line}")
                continue

            proxies.append({"http": proxy_url, "https": proxy_url})

    return proxies


def read_rows(csv_file):
    rows = []
    with open(csv_file, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(
                {
                    "ISRC": (row.get("ISRC") or "").strip(),
                    "youtube_id": (row.get("youtube_id") or "").strip(),
                }
            )
    return rows

def write_rows(csv_file, rows):
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["ISRC", "youtube_id"])
        writer.writeheader()
        writer.writerows(rows)


def parse_duration_seconds(duration_text):
    if not duration_text or duration_text == 0:
        return None

    text = str(duration_text).strip()
    if not text:
        return None

    parts = text.split(":")
    if not all(p.isdigit() for p in parts):
        return None

    if len(parts) == 3:
        h, m, s = map(int, parts)
        return h * 3600 + m * 60 + s
    if len(parts) == 2:
        m, s = map(int, parts)
        return m * 60 + s
    if len(parts) == 1:
        return int(parts[0])
    return None

# shorts are excluded
def is_short(video):
    suffix = str(video.get("url_suffix") or "")
    if suffix.startswith("/shorts/"):
        return True

    seconds = parse_duration_seconds(video.get("duration"))
    if seconds is not None and seconds <= 60:
        return True

    return False


def has_required_suffix(video):
    suffix = str(video.get("url_suffix") or "")
    return REQUIRED_URL_SUFFIX_TOKEN in suffix

# youtube videos that are music videos all have a publish_time of 0. 
# Videos with a different time are excluded.
def has_allowed_publish_time(video):
    p = video.get("publish_time")
    return p == 0 or str(p).strip() == "0"


def pick_best_video(results):
    if not results:
        return None

    filtered = []
    for v in results:
        if is_short(v):
            continue
        if not has_required_suffix(v):
            continue
        if not has_allowed_publish_time(v):
            continue
        filtered.append(v)

    if not filtered:
        return None

    filtered.sort(key=lambda x: parse_duration_seconds(x.get("duration")) or 0, reverse=True)
    return filtered[0]


def extract_youtube_id(video):
    if not video:
        return ""

    video_id = (video.get("id") or "").strip()
    if video_id:
        return video_id

    suffix = (video.get("url_suffix") or "").strip()
    if "v=" in suffix:
        return suffix.split("v=", 1)[1].split("&", 1)[0]
    if suffix.startswith("/watch/"):
        return suffix.rsplit("/", 1)[-1]
    return ""


def proxy_key(proxy):
    return proxy.get("http", "")


def choose_proxy(proxies):
    return random.choice([p for p in proxies])


def search_one(isrc, proxy, timeout, retries):
    results = YoutubeSearch(
        isrc,
        max_results=SEARCH_MAX_RESULTS,
        proxy=proxy,
        retries=retries,
        timeout=timeout,
    ).to_dict()

    best = pick_best_video(results)
    youtube_id = extract_youtube_id(best)
    return youtube_id, best


def run_batch(
    source_csv=SOURCE_CSV_PATH,
    proxy_file=PROXY_FILE,
    batch_size=BATCH_SIZE,
    max_workers=MAX_WORKERS,
    timeout=TIMEOUT,
    retries=RETRIES
):
    rows = read_rows(source_csv)

    proxies = load_proxies(proxy_file)
    if not proxies:
        raise RuntimeError("No valid proxies found in proxy file.")

    processed = 0

    idx_cursor = 0

    while idx_cursor < len(rows):
        wave = []
        while idx_cursor < len(rows) and len(wave) < max_workers and processed + len(wave) < batch_size:
            isrc = rows[idx_cursor]["ISRC"]
            if isrc:
                wave.append((idx_cursor, isrc))
            idx_cursor += 1

        if not wave:
            continue

        future_map = {}
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            for idx, isrc in wave:
                proxy = choose_proxy(proxies)
                key = proxy_key(proxy)
                fut = ex.submit(search_one, isrc, proxy, timeout, retries)
                future_map[fut] = (idx, isrc, key)

            for fut in as_completed(future_map):
                idx, isrc, pkey = future_map[fut]
                processed += 1

                try:
                    youtube_id, best_result = fut.result()
                    
                    if youtube_id:
                        rows[idx]["youtube_id"] = youtube_id

                except Exception as exc:
                    pass

    write_rows(source_csv, rows)

    return {"processed": processed}


def run_batches(max_batches=None):
    if not Path(SOURCE_CSV_PATH).exists():
        raise FileNotFoundError(f"csv not found: {SOURCE_CSV_PATH}")

    batches = 0
    while True:
        run_batch()
        batches += 1


# Run batches from existing csv
run_batches()

In [ ]:
ISRC_CSV = Path("isrcs.csv")


total = 0
matched = 0

with ISRC_CSV.open("r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        isrc = (row.get("ISRC") or "").strip()
        if not isrc:
            continue
        total += 1
        if (row.get("youtube_id") or "").strip():
            matched += 1

pct = (matched / total * 100.0) if total else 0.0

print(f"File: {ISRC_CSV}")
print(f"Total ISRC rows: {total}")
print(f"Matched youtube_id: {matched}")
print(f"Match rate: {pct:.2f}%")



File: isrcs.csv
Total ISRC rows: 38765
Matched youtube_id: 33661
Match rate: 86.83%
